# 1일차 파트 2 실습 풀이

`1일차_02_실습.ipynb`의 풀이와 해설입니다. 프롬프트는 하나의 예시입니다. 모델의 응답은 실행할 때마다 다르므로 여러분의 프롬프트와 결과가 풀이와 달라도 테스트를 통과하면 됩니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 받지 않을 때만 바꿉니다(2일차 파트 2의 "구조화 출력" 절) |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. Windows PowerShell에서는 다음과 같이 설정한 뒤 터미널을 새로 엽니다.

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. 이 값이 없으면 노트북은 Anthropic API를 쓰는 것으로 봅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 마지막 줄에 `준비 완료`가 나오면 연결된 것입니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data/1일차'


def read_material(name):
    """재료 폴더(data/1일차)의 파일을 읽어 문자열로 돌려줍니다."""
    return open(f'{DATA}/{name}', encoding='utf-8').read().strip()


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ... 가 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `401 auth` | 키가 틀렸습니다. 앞뒤 공백이나 따옴표 없이 다시 넣습니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `NameError` | 앞 셀을 건너뛰었습니다. 위에서부터 차례로 다시 실행합니다 |

## 실습 2 알람 로그를 분류하는 프롬프트 작성

**목적** 분류 기준, 역할 지정, 출력 형식, 예시를 넣은 프롬프트로 분류 결과가 실행마다 같아지는지 확인합니다.

**재료** `data/1일차/알람로그_10건.txt`(야간 설비 알람 로그 10건)와 `data/1일차/알람_분류기준.txt`(조치 필요로 분류하는 기준 세 가지)입니다. 줄별 정답은 `알람로그_정답.txt`에 있고, 테스트 셀이 읽습니다. 먼저 열어 보지 않고 풀어 봅니다.

**할 일**

1. 알람 로그와 분류 기준을 받아 프롬프트 문자열을 돌려주는 `build_classify_prompt()`를 작성합니다. 프롬프트에는 역할 지정, 맥락(누가 읽고 어디에 쓰는지), 분류 기준, 출력 형식이 들어가야 합니다.
2. 출력은 코드가 읽을 수 있게 한 줄에 하나씩 `번호 | 분류`로만 받습니다. 분류는 `조치 필요` 또는 `모니터링`입니다.
3. 테스트 셀을 실행합니다. 같은 프롬프트를 새 채팅에서 세 번 보내 응답을 검사합니다.
4. 테스트에서 실패한 항목이 있으면 프롬프트를 고쳐 다시 실행합니다. 분류가 바뀐 줄이 있으면 그 줄의 원문을 읽고, 분류 기준 어느 항목이 적용되는지 확인합니다.

**완료 기준** 테스트 셀이 "실습 2 통과"를 출력합니다. 세 번 모두 10건을 읽을 수 있고, 형식 외의 말이 없고, 세 번의 분류가 같고, 정답과 9건 이상 일치해야 합니다.

<details><summary>힌트 1</summary>

분류 기준 파일의 글을 프롬프트에 그대로 넣습니다. 기준을 말로 풀어 쓰면 모델이 기준을 다르게 읽을 수 있습니다.
</details>

<details><summary>힌트 2</summary>

세 번의 분류가 같지 않으면 분류가 바뀐 줄을 찾습니다. 그 줄과 비슷한 지어낸 줄을 Few-shot 예시로 한두 개 넣습니다. 로그의 줄을 그대로 예시로 넣으면 그 줄의 분류는 확인할 수 없으므로, 기준은 같고 내용이 다른 줄을 지어냅니다.
</details>

<details><summary>힌트 3</summary>

"다른 말, 설명, 코드 블록 표시는 쓰지 마"를 적지 않으면 응답 앞뒤에 인사말이나 표 머리글이 붙어 형식 검사에서 실패합니다.
</details>

In [3]:
# 실습 2 재료 (그대로 실행합니다)
import checks_d1_2

log = read_material('알람로그_10건.txt')
rule = read_material('알람_분류기준.txt')
print(log)
print()
print(rule)

[3라인 설비 알람 로그 · 10/2 야간조]
1. 02:14 5호기 E-201 펌프 압력 이상 (이번 주 3회째)
2. 02:40 2호기 챔버 압력 상승 경보 1회, 자동 복귀
3. 03:05 7호기 로봇 티칭 위치 편차 경고, 재시작 후 정상
4. 03:30 5호기 E-201 재발
5. 04:12 4호기 진공펌프 온도 상한 근접 (85도 / 상한 90도)
6. 04:50 1호기 도어 인터락 해제 감지, 작업자 확인 완료
7. 05:20 3호기 RF 반사파 증가 경고 2회
8. 05:45 6호기 칠러 유량 저하 경고, 원인 미상
9. 06:10 2호기 챔버 압력 상승 경보 재발
10. 06:30 8호기 통신 끊김 1분, 자동 재연결

조치 필요로 분류하는 기준은 다음 세 가지 가운데 하나라도 해당할 때입니다. 하나도 해당하지 않으면 모니터링입니다.

1. 같은 밤에 같은 설비에서 같은 알람이 두 번 이상 발생했다. 두 번째 줄뿐 아니라 첫 번째 줄도 조치 필요입니다. 한 줄 안에 횟수가 2회 이상으로 적힌 경우도 같습니다.
2. 측정값이 상한의 90% 이상이다.
3. 원인을 알 수 없다고 적혀 있다.

자동으로 복귀했거나 재시작 후 정상으로 돌아왔다는 사실은 위 세 기준을 바꾸지 않습니다.


In [4]:
# 실습 2 (1) 분류 프롬프트를 작성합니다
def build_classify_prompt(log, rule):
    """알람 로그와 분류 기준을 받아 모델에 보낼 프롬프트 문자열을 돌려줍니다."""
    return ('너는 반도체 장비팀 설비 엔지니어다. 이 자료는 아침 인수인계 때 주간조 파트장이 읽고 '
            '오늘 먼저 조치할 설비를 정하는 데 쓴다.\n'
            '다음 야간 설비 알람 로그를 [조치 필요 / 모니터링] 두 가지로 분류해줘.\n\n'
            + rule + '\n'
            '예시:\n'
            '- "9호기 칠러 유량 저하 경고, 원인 미상" → 조치 필요\n'
            '- "4호기 센서 오차 경고, 재시작 후 정상" → 모니터링\n\n'
            '출력은 한 줄에 하나씩 "번호 | 분류" 형식으로만 적고, 분류는 조치 필요 또는 모니터링 가운데 '
            '하나만 써. 다른 말, 설명, 코드 블록 표시는 쓰지 마.\n\n' + log)

In [5]:
# 실습 2 (2) 같은 프롬프트를 새 채팅에서 세 번 보내고 테스트합니다 (실제 API 를 부릅니다. 3회)
classify_prompt = build_classify_prompt(log, rule)
texts = []
for number in range(3):
    chat = llm.Chat(model='소형')
    chat.say(classify_prompt, show=False)
    texts.append(chat.last.text)
print(texts[0])
print()
checks_d1_2.check_classify(texts)

1 | 조치 필요
2 | 조치 필요
3 | 모니터링
4 | 조치 필요
5 | 조치 필요
6 | 모니터링
7 | 조치 필요
8 | 조치 필요
9 | 조치 필요
10 | 모니터링

1. 통과 세 번 모두 10건의 분류를 읽을 수 있다
2. 통과 응답에 "번호 | 분류" 줄 외의 말이나 코드 블록 표시가 없다
3. 통과 세 번의 분류가 모두 같다
4. 통과 세 번 모두 정답과 9건 이상 일치한다(일치한 건수: [10, 10, 10])
실습 2 통과


[(True, '세 번 모두 10건의 분류를 읽을 수 있다'),
 (True, '응답에 "번호 | 분류" 줄 외의 말이나 코드 블록 표시가 없다'),
 (True, '세 번의 분류가 모두 같다'),
 (True, '세 번 모두 정답과 9건 이상 일치한다(일치한 건수: [10, 10, 10])')]

## 실습 3 분류 결과에서 표를 만들고 개수를 확인하는 체인

**목적** 실습 2의 분류 결과를 1단으로 삼아 2단(표 만들기)을 보내고, 표의 행 수를 모델에게 묻지 않고 코드로 센 값으로 확인합니다. 제약 한 줄을 빼면 표가 어떻게 바뀌는지 비교합니다.

**재료** 실습 2의 `texts[0]`(1단의 분류 결과). 실습 2를 먼저 실행해야 합니다.

**할 일**

1. 마크다운 표의 데이터 줄 수를 세는 `count_table_rows()`를 작성합니다. 머리글 줄과 구분선(`| --- |`)은 세지 않고, 표 앞뒤의 문장도 세지 않습니다.
2. 2단 프롬프트 `STEP2_PROMPT`를 작성합니다. 조치 필요인 로그만 한 줄을 한 행으로 정리하고, 열은 `로그 번호 | 설비 | 알람 | 발생 시각`이며, 원자료에 없는 값은 `확인 필요`로 적게 합니다. 표 밖의 말은 쓰지 않게 합니다. `MERGE_BAN`의 문장을 그대로 프롬프트에 넣습니다.
3. 테스트 셀을 실행합니다. 1단 결과를 대화 이력에 넣고 2단을 보냅니다.
4. `MERGE_BAN`을 뺀 프롬프트를 두 번 보내 표의 행 수와 로그 번호가 어떻게 바뀌는지 봅니다.
5. 표를 루브릭 네 항목(빠짐없음, 정확함, 바로 쓸 수 있음, 목적 맞춤)으로 직접 채점합니다. 점수는 1~3점이고 기준은 교재의 예제 2-9를 따릅니다. 이번에는 모델에게 채점을 맡기지 않습니다.

**완료 기준** 테스트 셀이 "실습 3 통과"를 출력합니다. 표의 행 수와 로그 번호가 정답의 조치 필요 건과 같아야 합니다.

<details><summary>힌트 1</summary>

데이터 줄은 `|`로 시작하고 첫 칸이 숫자입니다. 정규식 `^\s*\|\s*\d+\s*\|`로 줄마다 확인하면 머리글과 구분선이 빠집니다.
</details>

<details><summary>힌트 2</summary>

같은 설비의 두 줄(5호기 E-201의 1번과 4번)이 한 행으로 합쳐지면 표의 행 수가 정답보다 적어집니다. 합치지 않는다는 문장을 `MERGE_BAN`으로 넣는 이유가 이것입니다.
</details>

In [6]:
# 실습 3 (1) 표의 데이터 줄 수를 세는 함수를 작성합니다
import re


def count_table_rows(text):
    """마크다운 표의 데이터 줄 수를 돌려줍니다. 머리글 줄, 구분선, 표 밖의 문장은 세지 않습니다."""
    return len([line for line in text.splitlines() if re.match(r'^\s*\|\s*\d+\s*\|', line)])

In [7]:
# 실습 3 (2) 2단 프롬프트를 작성합니다
MERGE_BAN = '같은 설비가 두 번 나와도 행을 합치지 마.'          # 이 문장을 그대로 프롬프트에 넣습니다
STEP2_PROMPT = ('위 [조치 필요]에서 로그 한 줄을 표 한 행으로 정리해줘. ' + MERGE_BAN + '\n'
                '열: 로그 번호 | 설비 | 알람 | 발생 시각\n'
                "원자료에 없으면 '확인 필요'로 적어줘. 표 외의 다른 말은 하지 마.")

In [8]:
# 실습 3 (3) 1단 결과를 대화 이력에 넣고 2단을 보내 테스트합니다 (실제 API 를 부릅니다. 1회)
def start_chain():
    """실습 2의 프롬프트와 첫 응답을 이력으로 가진 채팅을 돌려줍니다."""
    chat = llm.Chat(model='소형')
    chat.history = [{'role': 'user', 'content': classify_prompt},
                    {'role': 'assistant', 'content': texts[0]}]
    return chat


chat = start_chain()
chat.say(STEP2_PROMPT)
print()
checks_d1_2.check_chain(chat.last.text, count_table_rows, texts[0])

| 로그 번호 | 설비 | 알람 | 발생 시각 |
|---|---|---|---|
| 1 | 5호기 E-201 펌프 | 압력 이상 (이번 주 3회째) | 02:14 |
| 2 | 2호기 챔버 | 압력 상승 경보 (1회, 자동 복귀) | 02:40 |
| 4 | 5호기 E-201 펌프 | 확인 필요 (원자료에 '재발'만 기재) | 03:30 |
| 5 | 4호기 진공펌프 | 온도 상한 근접 (85도 / 상한 90도) | 04:12 |
| 7 | 3호기 RF | 반사파 증가 경고 (2회) | 05:20 |
| 8 | 6호기 칠러 | 유량 저하 경고 (원인 미상) | 05:45 |
| 9 | 2호기 챔버 | 압력 상승 경보 (재발) | 06:10 |

1. 통과 count_table_rows 가 머리글과 구분선을 빼고 데이터 줄 3개를 센다
2. 통과 표의 행 수가 조치 필요 7건과 같다(센 값: 7)
3. 통과 표의 로그 번호가 정답의 조치 필요 번호와 같다(표의 번호: [1, 2, 4, 5, 7, 8, 9])
4. 통과 표의 행 수가 1단 분류에서 센 조치 필요 건수와 같다(7건)
실습 3 통과


[(True, 'count_table_rows 가 머리글과 구분선을 빼고 데이터 줄 3개를 센다'),
 (True, '표의 행 수가 조치 필요 7건과 같다(센 값: 7)'),
 (True, '표의 로그 번호가 정답의 조치 필요 번호와 같다(표의 번호: [1, 2, 4, 5, 7, 8, 9])'),
 (True, '표의 행 수가 1단 분류에서 센 조치 필요 건수와 같다(7건)')]

In [9]:
# 실습 3 (4) 합치지 말라는 문장을 빼고 두 번 보냅니다 (실제 API 를 부릅니다. 2회)
for number in [1, 2]:
    chat = start_chain()
    chat.say(STEP2_PROMPT.replace(MERGE_BAN, ''), show=False)
    print(f'[{number}번째] 표의 행 수 {count_table_rows(chat.last.text)}, '
          f'로그 번호 {checks_d1_2.table_numbers(chat.last.text)}')

[1번째] 표의 행 수 7, 로그 번호 [1, 2, 4, 5, 7, 8, 9]


[2번째] 표의 행 수 7, 로그 번호 [1, 2, 4, 5, 7, 8, 9]


**해설**

아래는 2026-11-23에 Claude 모델로 실행한 결과입니다. 여러분의 응답은 다를 수 있습니다.

TODO: 풀이 실행 결과